# Multi-Objective Workforce Optimization with cuOpt Python API

This notebook demonstrates how to solve a **multi-objective** workforce optimization problem using the cuOpt Python API — exploring the tradeoffs among labor cost, coverage, and fairness instead of returning a single plan.

The base `workforce_optimization_milp` notebook minimizes labor cost with coverage **hard-constrained** — one plan, the cheapest way to fully staff. But a planner usually faces a **tradeoff with no fixed weighting**: *how much coverage is worth how much cost?* and *how much does fairness cost?* A single solve hides that — you get one point on a curve you can't see.

We turn that single solve into the **whole tradeoff curve** with the **ε-constraint method**:

1. **Recognize a hidden objective** — a hard constraint whose level was *assumed* rather than given is a candidate objective.
2. **Anchor** each objective's range (solve each on its own).
3. **Sweep** the promoted constraint as a parametric bound; each solve is one frontier point.
4. **Filter** dominated points and **read** the frontier — quote the exchange rate (extra $ per extra shift).

We run it twice, promoting a different "fixed" constraint each time:
- **cost vs. coverage** — relax `coverage == required` and sweep a coverage floor.
- **cost vs. fairness** — sweep the base model's fixed `max_shifts` cap.

(This workflow is also packaged as the `cuopt-multi-objective-exploration` skill.)

## Environment Setup

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>

            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>

            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>

            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
# Uncomment for your CUDA version if cuOpt is not already installed (e.g., Google Colab):
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu12  # CUDA 12
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu13  # CUDA 13

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cuopt.linear_programming.problem import Problem, VType, sense, LinearExpression
from cuopt.linear_programming.solver_settings import SolverSettings
print("Imports ready")

## Problem data

Same workers, shifts, pay, and availability as the base `workforce_optimization_milp` notebook.

In [ ]:
shift_requirements = {
    "Mon1": 3, "Tue2": 2, "Wed3": 4, "Thu4": 2, "Fri5": 5, "Sat6": 3, "Sun7": 4,
    "Mon8": 2, "Tue9": 2, "Wed10": 3, "Thu11": 4, "Fri12": 5, "Sat13": 7, "Sun14": 5,
}
worker_pay = {"Amy": 10, "Bob": 12, "Cathy": 10, "Dan": 8, "Ed": 8, "Fred": 9, "Gu": 11}
availability = {
    "Amy":   ["Tue2","Wed3","Fri5","Sun7","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Bob":   ["Mon1","Tue2","Fri5","Sat6","Mon8","Thu11","Sat13","Sun14"],
    "Cathy": ["Wed3","Thu4","Fri5","Sun7","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Dan":   ["Tue2","Wed3","Fri5","Sat6","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Ed":    ["Mon1","Tue2","Wed3","Thu4","Fri5","Sun7","Mon8","Tue9","Thu11","Sat13","Sun14"],
    "Fred":  ["Mon1","Tue2","Wed3","Sat6","Mon8","Tue9","Fri12","Sat13","Sun14"],
    "Gu":    ["Mon1","Tue2","Wed3","Fri5","Sat6","Sun7","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
}
pairs = [(w, s) for w, shifts in availability.items() for s in shifts]
TOTAL_REQUIRED = sum(shift_requirements.values())
print(f"{len(worker_pay)} workers, {len(shift_requirements)} shifts, {len(pairs)} feasible (worker,shift) pairs")
print(f"Full coverage = {TOTAL_REQUIRED} staffed shifts")

## A solver helper (one model, used for every point)

Binary `x[w,s]` for each available pair; `assigned[s] ≤ required[s]` (no overstaffing, which keeps *coverage* a clean linear count). The objective is labor cost; an optional **coverage floor** is the parametric ε-constraint we'll sweep. A `time_limit` bounds every MILP solve.

In [ ]:
def solve(coverage_floor=None, maximize_coverage=False, time_limit=10.0):
    """Solve one workforce MILP and return its cost, coverage, and status.

    Minimizes labor cost over the binary (worker, shift) assignment model (no
    overstaffing) -- or, with maximize_coverage, maximizes shifts staffed to anchor
    the sweep. An optional coverage_floor adds the epsilon-constraint
    coverage >= coverage_floor, swept to trace the cost-vs-coverage frontier.

    Parameters
    ----------
    coverage_floor : int, optional
        Minimum shifts that must be staffed (the swept epsilon-constraint); None = no floor.
    maximize_coverage : bool, default False
        Maximize coverage instead of minimizing cost (anchors the sweep range).
    time_limit : float, default 10.0
        Per-solve time limit in seconds, guarding branch-and-bound.

    Returns
    -------
    dict or None
        {"cost", "coverage", "status"} for an Optimal/FeasibleFound solve, else None.
    """
    prob = Problem("workforce")
    x = {p: prob.addVariable(name=f"{p[0]}_{p[1]}", vtype=VType.INTEGER, lb=0.0, ub=1.0) for p in pairs}
    obj = LinearExpression([], [], 0.0)
    for (w, s), var in x.items():
        coef = (-1.0) if maximize_coverage else float(worker_pay[w])   # maximize coverage = minimize -sum(x)
        if coef != 0:
            obj += var * coef
    prob.setObjective(obj, sense.MINIMIZE)
    for s, req in shift_requirements.items():                          # no overstaffing
        e = LinearExpression([], [], 0.0); has = False
        for (w, s2), var in x.items():
            if s2 == s:
                e += var; has = True
        if has:
            prob.addConstraint(e <= req, name=f"cap_{s}")
    if coverage_floor is not None:                                     # epsilon-constraint
        cov = LinearExpression([], [], 0.0)
        for var in x.values():
            cov += var
        prob.addConstraint(cov >= float(coverage_floor), name="coverage_floor")
    settings = SolverSettings()
    settings.set_parameter("time_limit", float(time_limit))
    settings.set_parameter("log_to_console", False)
    prob.solve(settings)
    if prob.Status.name not in ("Optimal", "FeasibleFound"):
        return None
    sel = [(w, s) for (w, s), var in x.items() if var.getValue() > 0.5]
    return {"cost": sum(worker_pay[w] for (w, s) in sel), "coverage": len(sel), "status": prob.Status.name}

## One objective → one plan (the base model)

The base notebook minimizes cost at **full** coverage. That's a single point: the cheapest way to staff everything.

In [ ]:
base = solve(coverage_floor=TOTAL_REQUIRED)
print(f"Cheapest full-coverage plan: cover {base['coverage']}/{TOTAL_REQUIRED} shifts at ${base['cost']}  ({base['status']})")
print("That's one point. Is full coverage worth its cost vs. covering a little less? One solve can't say.")

## Two objectives, no fixed weighting → trace the frontier

**Anchor** the objectives (coverage ranges 0…max; cost 0…full-coverage cost), then **ε-constraint sweep** — minimize cost subject to `coverage ≥ ε`, for ε across the range — and **filter** to the non-dominated set.

In [ ]:
cov_max = solve(maximize_coverage=True)["coverage"]
points, non_optimal = [], 0
for eps in range(0, cov_max + 1):
    r = solve(coverage_floor=eps)
    if r:
        points.append((r["coverage"], r["cost"]))
        if r["status"] != "Optimal":  # solved only to the gap within the time limit
            non_optimal += 1

def non_dominated(pts):                       # maximize coverage, minimize cost
    return sorted({(c, k) for (c, k) in pts
                   if not any((c2 >= c and k2 <= k and (c2 > c or k2 < k)) for (c2, k2) in pts)})

frontier = non_dominated(points)
print(f"Max achievable coverage: {cov_max}/{TOTAL_REQUIRED} | frontier points: {len(frontier)}")
print(f"Swept solves: {len(points)} | not certified-Optimal (FeasibleFound): {non_optimal}")

In [ ]:
fr = np.array(frontier)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(fr[:, 0], fr[:, 1], "o-", color="navy", lw=1.6, label=f"cost-vs-coverage frontier ({len(frontier)} options)")
ax.scatter([base["coverage"]], [base["cost"]], s=240, marker="*", color="crimson", zorder=5,
           label="base model: one full-coverage plan")
ax.set_xlabel("Coverage (shifts staffed)"); ax.set_ylabel("Labor cost ($)")
ax.set_title("One solve is one point; the frontier is the whole decision")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Read the frontier — this is the value

The interpretation step: quote the **exchange rate** (extra $ per extra shift covered) between adjacent points, so the planner can decide *where on the curve* to sit. No single "best" — it's a choice the frontier makes visible.

In [ ]:
print("Marginal cost of coverage along the frontier:")
for i in range(1, len(frontier)):
    dcov = frontier[i][0] - frontier[i-1][0]
    dcost = frontier[i][1] - frontier[i-1][1]
    if dcov:
        print(f"  coverage {frontier[i-1][0]:2d} -> {frontier[i][0]:2d}:  +${dcost} for +{dcov} shift  (${dcost/dcov:.0f}/shift)")
print("\nThe single solve only ever showed the right-most point. The frontier shows the price of every coverage level.")

**Method note.** We use the default, **ε-constraint** (minimize one objective, sweep the others as bounds): it enumerates every efficient point and stays correct when the frontier is non-convex. A weighted-sum sweep would agree on the supported points of this (convex) frontier, but on non-convex problems — common in combinatorial MILPs — it can skip efficient points entirely, which is why ε-constraint is the default.

## A second tradeoff, for free — cost vs. fairness

The base model *fixed* `max_shifts_per_worker = 4`. The recognition move — **a fixed constraint whose assumed level is a candidate objective** — says: sweep that cap instead of fixing it. A tighter cap spreads work more evenly (fairer) but costs more. Same ε-constraint mechanic, a different tradeoff, no new data.

In [ ]:
def solve_fairness(max_shifts, time_limit=10.0):
    """Solve the full-coverage workforce MILP under a per-worker shift cap.

    Sweeping max_shifts promotes the base model's fixed cap to an epsilon-constraint,
    tracing the cost-vs-fairness frontier: a tighter cap spreads work more evenly
    (fairer) but costs more.

    Parameters
    ----------
    max_shifts : int
        Maximum shifts any single worker may take (the swept fairness lever).
    time_limit : float, default 10.0
        Per-solve time limit in seconds.

    Returns
    -------
    dict or None
        {"max_shifts", "cost", "busiest", "status"} if feasible, else None.
    """
    prob = Problem("workforce_fairness")
    x = {p: prob.addVariable(name=f"{p[0]}_{p[1]}", vtype=VType.INTEGER, lb=0.0, ub=1.0) for p in pairs}
    obj = LinearExpression([], [], 0.0)
    for (w, s), var in x.items():
        if worker_pay[w]:
            obj += var * worker_pay[w]
    prob.setObjective(obj, sense.MINIMIZE)
    for s, req in shift_requirements.items():                 # full coverage (hard)
        e = LinearExpression([], [], 0.0); has = False
        for (w, s2), var in x.items():
            if s2 == s:
                e += var; has = True
        if has:
            prob.addConstraint(e == req, name=f"cover_{s}")
    for w in worker_pay:                                      # fairness lever: per-worker cap
        e = LinearExpression([], [], 0.0); has = False
        for (w2, s), var in x.items():
            if w2 == w:
                e += var; has = True
        if has:
            prob.addConstraint(e <= float(max_shifts), name=f"cap_{w}")
    settings = SolverSettings(); settings.set_parameter("time_limit", float(time_limit)); settings.set_parameter("log_to_console", False)
    prob.solve(settings)
    if prob.Status.name not in ("Optimal", "FeasibleFound"):
        return None
    sel = [(w, s) for (w, s), var in x.items() if var.getValue() > 0.5]
    busiest = max((sum(1 for (w2, s) in sel if w2 == w) for w in worker_pay), default=0)
    return {"max_shifts": max_shifts, "cost": sum(worker_pay[w] for (w, s) in sel), "busiest": busiest, "status": prob.Status.name}

fair, non_optimal = [], 0
for cap in range(len(shift_requirements), 0, -1):
    r = solve_fairness(cap)
    print(f"max_shifts cap {cap:2d}: " + (f"full coverage at ${r['cost']}, busiest worker {r['busiest']} shifts" if r else "INFEASIBLE (cap too tight to staff every shift)"))
    if r:
        fair.append((r["max_shifts"], r["cost"]))
        if r["status"] != "Optimal":
            non_optimal += 1
print(f"Feasible caps: {len(fair)} | not certified-Optimal (FeasibleFound): {non_optimal}")

if fair:
    fp = np.array(fair)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(fp[:, 0], fp[:, 1], "o-", color="seagreen", lw=1.6)
    ax.invert_xaxis()
    ax.set_xlabel("Max shifts per worker  (left = fairer)"); ax.set_ylabel("Labor cost ($) at full coverage")
    ax.set_title("cost vs. fairness: the price of spreading work evenly")
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Notes & takeaway

**Takeaway — reusing this on your own problem.** When a single-objective model has a hard constraint whose level was *assumed* — a coverage target, a per-resource cap, a budget — that constraint is a hidden objective. Promote it to a swept ε-constraint, collect the non-dominated points, and read the exchange rate off the frontier. The same recipe traces any such tradeoff.

- **Synthetic data** — the base notebook's toy roster; this demonstrates the *method*, not a staffing study.
- **Optimal to the gap** — each point is solved under a `time_limit`; every solve here returned `Optimal` (0 `FeasibleFound`), so each is optimal to cuOpt's MIP gap (exact here, since labor cost is integer-valued).
- **No duals for a MILP** — an integer program has no constraint duals, so the marginal cost of coverage is read off the frontier itself (above).

## License

SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.